In [1]:
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
import torch

from transformers import AutoTokenizer, M2M100ForConditionalGeneration, GenerationConfig, BitsAndBytesConfig
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from pythainlp.tokenize import word_tokenize
from jiwer import wer

import pandas as pd
import numpy as np

from tqdm import tqdm
import json

ROOT_DIR = "/project/lt200304-dipmt/paweekorn"

INFO 09-23 13:17:57 [__init__.py:241] Automatically detected platform cuda.


## Overview

In [2]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# dataset preparation
test_df = pd.read_excel('/project/lt200304-dipmt/paweekorn/data/test_set.xlsx')

print(test_df.shape)
test_df.head()

(8392, 4)


,ID,NAME,ENG,THA
0,1,9,"sockets, plugs and other contacts (electric co...",เต้ารับ ปลั๊กและตัวติดต่อ (ตัวเชื่อมต่อไฟฟ้า)
1,2,9,"material for electricity mains (wires, cables)",วัสดุสำหรับส่วนควบคุมไฟฟ้าหลัก (ลวด สายเคเบิล)
2,3,9,electrical instrument element and electrical i...,ส่วนประกอบอุปกรณ์ไฟฟ้าและวัสดุที่เฉพาะกับอุปกร...
3,4,12,"parts and accessories for automobiles, in part...",ชิ้นส่วนและอุปกรณ์เสริมสำหรับยานยนต์ โดยเฉพาะอ...
4,5,42,industrial analysis and research services.,บริการวิเคราะห์และวิจัยทางอุตสาหกรรม


## Inference

In [5]:
SRC_LANG  = "eng_Latn"
TGT_LANG  = "tha_Thai"
BATCH_SIZE = 64
MODEL_DIR = f"{ROOT_DIR}/models/fine-tuned/nllb-3.3b"

# ====== LOAD MODEL/TOKENIZER ======
model = M2M100ForConditionalGeneration.from_pretrained(MODEL_DIR, torch_dtype=torch.bfloat16, quantization_config=bnb_config)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
tokenizer.src_lang, tokenizer.tgt_lang = SRC_LANG, TGT_LANG

# NLLB needs forced BOS of target language
forced_bos = tokenizer.convert_tokens_to_ids(TGT_LANG)
model.config.forced_bos_token_id = forced_bos
model.config.use_cache = True

/home/psoratya/.conda/envs/unsloth_env/lib/python3.12/site-packages/transformers/quantizers/auto.py:231: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


In [ ]:
# ====== BATCHED GENERATION ======
def batched(iterable, n):
    for i in range(0, len(iterable), n):
        yield iterable[i:i+n]

# ====== GENERATION CONFIG ======
gen_cfg = GenerationConfig(
    max_length=1024,
    do_sample=True,
    num_beams=3,
    repetition_penalty=1.15,
    pad_token_id=1,
    bos_token_id=0,
    eos_token_id=2,
    decoder_start_token_id=2
)

# ====== Inference ======
preds = []
model.eval()
with torch.inference_mode():
    for batch_texts in tqdm(list(batched(test_df["ENG"].tolist(), BATCH_SIZE))):
        enc = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True).to(model.device)
        out = model.generate(**enc, generation_config=gen_cfg)
        batch_preds = tokenizer.batch_decode(out, skip_special_tokens=True)
        preds.extend(batch_preds)

# ====== WRITE RESULTS ======
test_df["PRED_THA"] = preds
print(preds[0])

  5%|▌         | 7/132 [00:32<09:47,  4.70s/it]


KeyboardInterrupt: 

In [16]:
wer_result, bleu = [], []
chencherry = SmoothingFunction().method1
for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    sol = row["THA"]
    pred = "" if pd.isna(row["PRED_THA"]) else row["PRED_THA"]

    # bleu score
    ref = word_tokenize(sol, engine='attacut')
    hyp = word_tokenize(pred, engine='attacut')
    
    wer_result.append(wer(sol, pred))
    bleu.append(sentence_bleu([ref], hyp, smoothing_function=chencherry))

print(f"Average WER:", np.mean(wer_result).round(4))
print(f"Average BLEU:", np.mean(bleu).round(4))

100%|██████████| 8392/8392 [00:46<00:00, 180.12it/s]

Average WER: 1.1611
Average BLEU: 0.2013
